# Excel → PowerPoint Converter

Copies selected tabs from an Excel workbook into a PowerPoint presentation.  
Each configured sheet becomes one slide. Supports **tables**, **charts**, and **mixed** content.

**Requires:**
```
pip install openpyxl python-pptx pandas matplotlib Pillow
```

In [ ]:
# Uncomment to install dependencies
# !pip install openpyxl python-pptx pandas matplotlib Pillow

## ⚙️ Configuration

Edit the values in this cell before running.

In [ ]:
# ─────────────────────────────────────────────────────────
# FILE PATHS
# ─────────────────────────────────────────────────────────
EXCEL_PATH  = "input.xlsx"   # Path to your Excel file
OUTPUT_PPTX = "output.pptx"  # Where to write the PowerPoint

# ─────────────────────────────────────────────────────────
# SHEET CONFIGURATION
#
# Required keys:
#   sheet_name   – exact name of the Excel tab
#
# Optional keys:
#   slide_title  – title shown on the slide (defaults to sheet_name)
#   content_type – "table" | "chart" | "mixed" | "auto" (default)
#                   auto  → table if no charts found, chart if charts found
#                   mixed → renders both table AND chart(s) on the same slide
#   data_range   – e.g. "A1:H30"; None = use Excel's used-range
#   header_row   – True (default) means the first data row is a header
#   chart_index  – which chart to use if sheet has multiple (0-indexed, default 0)
# ─────────────────────────────────────────────────────────
SHEET_CONFIG = [
    {
        "sheet_name":   "Sales Summary",
        "slide_title":  "Sales Summary",
        "content_type": "table",
        "data_range":   None,
        "header_row":   True,
    },
    {
        "sheet_name":   "Revenue Chart",
        "slide_title":  "Revenue Analysis",
        "content_type": "chart",
        "chart_index":  0,
    },
    {
        "sheet_name":   "KPIs",
        "slide_title":  "Key Performance Indicators",
        "content_type": "auto",
        "data_range":   None,
        "header_row":   True,
    },
]

# ─────────────────────────────────────────────────────────
# SLIDE THEME  (RGB tuples)
# ─────────────────────────────────────────────────────────
THEME = {
    "background":   (0xFF, 0xFF, 0xFF),  # slide background
    "title_color":  (0x1E, 0x27, 0x61),  # slide title text
    "accent":       (0x1E, 0x27, 0x61),  # separator line / header bg
    "header_text":  (0xFF, 0xFF, 0xFF),  # table header text
    "row_even":     (0xCA, 0xDC, 0xFC),  # table even-row fill
    "row_odd":      (0xFF, 0xFF, 0xFF),  # table odd-row fill
    "cell_text":    (0x1E, 0x1E, 0x1E),  # table body text
    "title_font":   "Calibri",
    "body_font":    "Calibri",
    # Chart series colors (hex strings, no #)
    "chart_colors": ["1E2761", "028090", "7A2048", "02C39A", "F96167", "F9E795"],
}

## 📦 Imports

In [ ]:
import os
import io
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import openpyxl
from openpyxl.utils import get_column_letter

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_AUTO_SHAPE_TYPE
from pptx.enum.dml import MSO_THEME_COLOR

print("✓ Libraries loaded")

## 🔧 Helper Functions — Tables

In [ ]:
def read_table_data(sheet, data_range=None):
    """
    Read cell values from a worksheet into a list-of-lists.
    Trailing all-None rows and columns are stripped.
    If data_range is given (e.g. 'A1:H30') only that area is read.
    """
    if data_range:
        cells = sheet[data_range]
        data = [[cell.value for cell in row] for row in cells]
    else:
        if sheet.max_row is None:
            return []
        data = [
            [cell.value for cell in row]
            for row in sheet.iter_rows(
                min_row=sheet.min_row,
                max_row=sheet.max_row,
                min_col=sheet.min_column,
                max_col=sheet.max_column,
            )
        ]

    # Strip trailing empty rows
    while data and all(v is None for v in data[-1]):
        data.pop()

    if not data:
        return []

    # Normalize row lengths
    max_cols = max(len(row) for row in data)
    data = [row + [None] * (max_cols - len(row)) for row in data]

    # Strip trailing empty columns
    while max_cols > 0 and all(row[max_cols - 1] is None for row in data):
        data = [row[:-1] for row in data]
        max_cols -= 1

    return data


def format_cell_value(value):
    """Format a cell value as a display string."""
    if value is None:
        return ""
    if isinstance(value, float):
        if value == int(value) and abs(value) < 1e12:
            return f"{int(value):,}"
        return f"{value:,.2f}"
    if isinstance(value, int):
        return f"{value:,}"
    return str(value)


def add_table_slide(
    slide, data, config, slide_w, slide_h, title_bottom, theme
):
    """
    Render a list-of-lists as a formatted python-pptx table.
    The table is auto-sized to fill the slide below the title.
    """
    if not data:
        return

    n_rows = len(data)
    n_cols = len(data[0])
    header = config.get("header_row", True)

    margin   = Inches(0.4)
    tbl_top  = title_bottom + Inches(0.15)
    tbl_w    = slide_w - 2 * margin
    tbl_h    = slide_h - tbl_top - margin

    # Clamp row height
    row_h = max(Pt(14), min(Inches(0.45), tbl_h // n_rows))
    actual_h = row_h * n_rows

    tbl_shape = slide.shapes.add_table(
        n_rows, n_cols, margin, tbl_top, tbl_w, actual_h
    )
    tbl = tbl_shape.table

    # Distribute column widths evenly
    col_w = tbl_w // n_cols
    for c in range(n_cols):
        tbl.columns[c].width = col_w

    # Dynamic font sizing
    if n_rows > 25 or n_cols > 10:
        body_fs, hdr_fs = Pt(7), Pt(8)
    elif n_rows > 15 or n_cols > 7:
        body_fs, hdr_fs = Pt(9), Pt(10)
    else:
        body_fs, hdr_fs = Pt(11), Pt(12)

    accent  = RGBColor(*theme["accent"])
    hdr_txt = RGBColor(*theme["header_text"])
    even_bg = RGBColor(*theme["row_even"])
    odd_bg  = RGBColor(*theme["row_odd"])
    body_c  = RGBColor(*theme["cell_text"])

    for r, row_data in enumerate(data):
        is_header = header and r == 0
        bg = accent if is_header else (even_bg if r % 2 == 0 else odd_bg)
        fs = hdr_fs if is_header else body_fs

        for c, value in enumerate(row_data):
            cell = tbl.cell(r, c)
            cell.text = format_cell_value(value)

            cell.fill.solid()
            cell.fill.fore_color.rgb = bg

            tf = cell.text_frame
            tf.word_wrap = False
            for para in tf.paragraphs:
                para.alignment = PP_ALIGN.LEFT
                for run in para.runs:
                    run.font.size  = fs
                    run.font.bold  = is_header
                    run.font.name  = theme["body_font"]
                    run.font.color.rgb = hdr_txt if is_header else body_c

    print(f"    Table: {n_rows} rows × {n_cols} cols")

## 🔧 Helper Functions — Charts

In [ ]:
def _resolve_ref(workbook, ref_str):
    """
    Resolve an openpyxl cell-range reference such as
    "Sheet1!$A$1:$D$10" or "'My Sheet'!B2:B20" into a flat
    list of non-None values.
    """
    if not ref_str or "!" not in ref_str:
        return []

    sheet_part, cell_part = ref_str.split("!", 1)
    sheet_name = sheet_part.strip("'\"")
    cell_range = cell_part.replace("$", "")

    try:
        ws = workbook[sheet_name]
    except KeyError:
        return []

    try:
        cells = ws[cell_range]
    except Exception:
        return []

    values = []
    # cells can be a tuple-of-tuples (2-D) or a single Cell
    if hasattr(cells, "value"):          # single cell
        if cells.value is not None:
            values.append(cells.value)
    else:
        for item in cells:
            if hasattr(item, "value"):   # 1-D
                if item.value is not None:
                    values.append(item.value)
            else:                        # 2-D
                for cell in item:
                    if cell.value is not None:
                        values.append(cell.value)
    return values


def _series_categories(series, workbook):
    """Return category labels for an openpyxl series, or None."""
    cat = getattr(series, "cat", None)
    if cat is None:
        return None
    for attr in ("numRef", "strRef"):
        ref_obj = getattr(cat, attr, None)
        if ref_obj and ref_obj.ref:
            vals = _resolve_ref(workbook, ref_obj.ref)
            if vals:
                return [str(v) for v in vals]
    return None


def _series_values(series, workbook):
    """Return numeric values for an openpyxl series, or None."""
    val = getattr(series, "val", None)
    if val is None:
        # ScatterChart uses xVal/yVal
        val = getattr(series, "yVal", None)
    if val is None:
        return None
    for attr in ("numRef", "strRef"):
        ref_obj = getattr(val, attr, None)
        if ref_obj and ref_obj.ref:
            raw = _resolve_ref(workbook, ref_obj.ref)
            if raw:
                try:
                    return [float(v) if v is not None else 0.0 for v in raw]
                except (TypeError, ValueError):
                    return [0.0] * len(raw)
    return None


def _series_title(series):
    """Return a string title for an openpyxl series, or None."""
    title = getattr(series, "title", None)
    if title is None:
        return None
    if hasattr(title, "v"):
        return str(title.v)
    if hasattr(title, "strRef"):
        try:
            cache = title.strRef.strCache
            if cache and cache.pt:
                return str(cache.pt[0].v)
        except Exception:
            pass
    return None


def _chart_title(xl_chart):
    """Return the chart title string, or None."""
    t = getattr(xl_chart, "title", None)
    if t is None:
        return None
    if isinstance(t, str):
        return t
    try:
        for para in t.tx.rich.paragraphs:
            for run in para.runs:
                if run.text:
                    return run.text
    except Exception:
        pass
    return None


def _chart_anchor_inches(xl_chart):
    """Return (width_in, height_in) from the chart's anchor, with fallback."""
    try:
        anchor = xl_chart.anchor
        # TwoCellAnchor has no direct ext — compute from to/from
        if hasattr(anchor, "ext"):
            return anchor.ext.cx / 914400, anchor.ext.cy / 914400
    except Exception:
        pass
    return 8.0, 4.5   # sensible default


# Map openpyxl class names → plot type strings
_CHART_TYPE_MAP = {
    "BarChart":     "bar",
    "LineChart":    "line",
    "PieChart":     "pie",
    "DoughnutChart":"donut",
    "AreaChart":    "area",
    "ScatterChart": "scatter",
    "RadarChart":   "radar",
}


def render_chart_to_bytes(workbook, sheet_name, chart_index=0, dpi=150, theme=None):
    """
    Extract chart data from an openpyxl chart and re-render it with
    matplotlib.  Returns (png_bytes_io, width_inches, height_inches)
    or None if the sheet has no charts / the chart cannot be rendered.
    """
    if theme is None:
        theme = {"chart_colors": ["1E2761","028090","7A2048","02C39A","F96167"]}

    ws = workbook[sheet_name]
    charts = getattr(ws, "_charts", [])
    if not charts:
        return None

    chart_index = min(chart_index, len(charts) - 1)
    xl_chart = charts[chart_index]
    chart_type = _CHART_TYPE_MAP.get(type(xl_chart).__name__, "bar")

    colors = [f"#{c}" for c in theme["chart_colors"]]
    w_in, h_in = _chart_anchor_inches(xl_chart)

    fig, ax = plt.subplots(figsize=(w_in, h_in))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    has_data = False
    categories = None

    for i, series in enumerate(xl_chart.series):
        color = colors[i % len(colors)]
        values = _series_values(series, workbook)
        if values is None:
            continue

        if categories is None:
            categories = _series_categories(series, workbook)

        label = _series_title(series) or f"Series {i+1}"
        x = categories if categories and len(categories) == len(values) else range(len(values))

        if chart_type == "bar":
            bar_dir = getattr(xl_chart, "barDir", "col")
            if bar_dir == "bar":
                ax.barh(x, values, label=label, color=color, alpha=0.85)
            else:
                ax.bar(x, values, label=label, color=color, alpha=0.85)

        elif chart_type == "line":
            ax.plot(x, values, label=label, color=color, linewidth=2,
                    marker="o", markersize=4)

        elif chart_type in ("pie", "donut"):
            pie_colors = [f"#{c}" for c in theme["chart_colors"][:len(values)]]
            wedge_props = {"linewidth": 1, "edgecolor": "white"}
            if chart_type == "donut":
                wedge_props["width"] = 0.5
            ax.pie(
                values,
                labels=categories[:len(values)] if categories else None,
                colors=pie_colors,
                autopct="%1.1f%%",
                startangle=90,
                wedgeprops=wedge_props,
            )
            ax.axis("equal")
            has_data = True
            break  # pie only uses one series

        elif chart_type == "area":
            xi = range(len(values))
            ax.fill_between(xi, values, alpha=0.45, color=color, label=label)
            ax.plot(xi, values, color=color, linewidth=1.5)

        elif chart_type == "scatter":
            ax.scatter(range(len(values)), values, label=label, color=color, s=40)

        else:  # fallback: line
            ax.plot(x, values, label=label, color=color, linewidth=2)

        has_data = True

    if not has_data:
        plt.close(fig)
        return None

    # ── Styling ──────────────────────────────────────────────────────
    if chart_type not in ("pie", "donut"):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        for spine in ("left", "bottom"):
            ax.spines[spine].set_color("#CCCCCC")
        ax.tick_params(colors="#64748B", labelsize=9)
        ax.yaxis.grid(True, color="#E2E8F0", linewidth=0.5, zorder=0)
        ax.set_axisbelow(True)
        if len(xl_chart.series) > 1:
            ax.legend(loc="upper right", frameon=False, fontsize=9)
        # Rotate x-labels if many categories
        if categories and len(categories) > 8:
            ax.set_xticks(range(len(categories)))
            ax.set_xticklabels(categories, rotation=35, ha="right", fontsize=8)

    title_str = _chart_title(xl_chart)
    if title_str:
        ax.set_title(title_str, fontsize=13, color="#1E2761", fontweight="bold", pad=10)

    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight",
                facecolor="white", edgecolor="none")
    buf.seek(0)
    plt.close(fig)

    return buf, w_in, h_in


def add_chart_to_slide(slide, img_bytes, img_w, img_h,
                       slide_w, slide_h, title_bottom):
    """
    Embed a PNG image (BytesIO) on the slide, centred below the title
    and scaled to fill the available area while preserving aspect ratio.
    """
    margin = Inches(0.4)
    avail_w = slide_w - 2 * margin
    avail_h = slide_h - title_bottom - 2 * margin

    aspect = img_w / img_h if img_h > 0 else 16 / 9

    if avail_w / aspect <= avail_h:
        disp_w = avail_w
        disp_h = int(disp_w / aspect)
    else:
        disp_h = avail_h
        disp_w = int(disp_h * aspect)

    left = (slide_w - disp_w) // 2
    top  = title_bottom + margin + (avail_h - disp_h) // 2

    slide.shapes.add_picture(img_bytes, left, top, disp_w, disp_h)
    print(f"    Chart: {img_w:.1f}" + '"' + f" × {img_h:.1f}" + '" (source)'
          f" → scaled to fit slide")

## 🔧 Helper Functions — Slide Frame

In [ ]:
def _blank_slide(prs):
    """Return a new blank slide (layout index 6)."""
    # Prefer the blank layout; fall back to the last layout if missing
    layouts = prs.slide_layouts
    blank = next(
        (l for l in layouts if l.name.lower() == "blank"),
        layouts[min(6, len(layouts) - 1)],
    )
    return prs.slides.add_slide(blank)


def add_slide_frame(slide, title_text, slide_w, slide_h, theme):
    """
    Add a white background, a bold title, and a thin accent separator.
    Returns the y-position (Emu) at which body content should begin.
    """
    # Background
    bg = slide.background
    bg.fill.solid()
    bg.fill.fore_color.rgb = RGBColor(*theme["background"])

    margin = Inches(0.4)
    title_h = Inches(0.65)

    # Title text box
    txb = slide.shapes.add_textbox(
        margin, Inches(0.12), slide_w - 2 * margin, title_h
    )
    tf = txb.text_frame
    tf.text = title_text
    p = tf.paragraphs[0]
    p.font.bold  = True
    p.font.size  = Pt(28)
    p.font.color.rgb = RGBColor(*theme["title_color"])
    p.font.name  = theme["title_font"]

    # Accent separator line (thin rectangle)
    sep_top  = Inches(0.82)
    sep_h    = Pt(2)
    sep      = slide.shapes.add_shape(
        MSO_AUTO_SHAPE_TYPE.RECTANGLE,
        margin, sep_top, slide_w - 2 * margin, sep_h,
    )
    sep.fill.solid()
    sep.fill.fore_color.rgb = RGBColor(*theme["accent"])
    sep.line.fill.background()   # no visible border

    return sep_top + sep_h + Inches(0.05)   # content starts here


def detect_content_type(ws):
    """Auto-detect whether a sheet is primarily a table, chart, or mixed."""
    has_charts = bool(getattr(ws, "_charts", []))
    # Consider a sheet to have tabular data if it has > 1 non-empty row
    has_table  = (ws.max_row or 0) > 1
    if has_charts and has_table:
        return "mixed"
    if has_charts:
        return "chart"
    return "table"

## ▶️ Main — Build the Presentation

In [ ]:
def build_presentation(excel_path, output_path, sheet_config, theme):
    """
    Core routine.  Iterates SHEET_CONFIG and creates one slide per entry.
    """
    print(f"Opening: {excel_path}")
    wb = openpyxl.load_workbook(excel_path, data_only=True)
    available = wb.sheetnames
    print(f"Available sheets: {available}\n")

    prs = Presentation()
    # Widescreen 16:9
    prs.slide_width  = Inches(13.33)
    prs.slide_height = Inches(7.5)

    slide_w = prs.slide_width
    slide_h = prs.slide_height

    for cfg in sheet_config:
        sheet_name  = cfg["sheet_name"]
        slide_title = cfg.get("slide_title", sheet_name)
        content_type = cfg.get("content_type", "auto")

        if sheet_name not in available:
            print(f"  ⚠  Sheet '{sheet_name}' not found — skipping.")
            continue

        print(f"Processing '{sheet_name}' …")
        ws = wb[sheet_name]

        # Resolve 'auto'
        if content_type == "auto":
            content_type = detect_content_type(ws)
            print(f"    Auto-detected: {content_type}")

        slide        = _blank_slide(prs)
        content_top  = add_slide_frame(slide, slide_title, slide_w, slide_h, theme)

        # ── CHART ─────────────────────────────────────────────────────
        if content_type in ("chart", "mixed"):
            chart_idx = cfg.get("chart_index", 0)
            result = render_chart_to_bytes(wb, sheet_name, chart_idx, theme=theme)
            if result:
                img_bytes, img_w, img_h = result
                if content_type == "mixed":
                    # Place chart on right half, table on left
                    half_w = slide_w // 2 - Inches(0.3)
                    margin = Inches(0.4)
                    avail_h = slide_h - content_top - margin
                    aspect  = img_w / img_h if img_h > 0 else 1.5
                    disp_h  = min(avail_h, int(half_w / aspect))
                    disp_w  = int(disp_h * aspect)
                    chart_left = slide_w // 2 + Inches(0.1)
                    chart_top  = content_top + (avail_h - disp_h) // 2
                    slide.shapes.add_picture(
                        img_bytes, chart_left, chart_top, disp_w, disp_h
                    )
                    print(f"    Chart (mixed): placed on right half")
                else:
                    add_chart_to_slide(
                        slide, img_bytes, img_w, img_h,
                        slide_w, slide_h, content_top,
                    )
            else:
                print(f"    ⚠  No renderable chart found in '{sheet_name}'")

        # ── TABLE ─────────────────────────────────────────────────────
        if content_type in ("table", "mixed"):
            data_range = cfg.get("data_range")
            data = read_table_data(ws, data_range)
            if data:
                if content_type == "mixed":
                    # Narrow the table to the left half
                    half_w = slide_w // 2 - Inches(0.5)
                    margin = Inches(0.4)

                    n_rows, n_cols = len(data), len(data[0])
                    row_h  = max(Pt(14), min(Inches(0.45),
                                 (slide_h - content_top - margin) // n_rows))
                    tbl_h  = row_h * n_rows

                    tbl_shape = slide.shapes.add_table(
                        n_rows, n_cols,
                        margin, content_top + Inches(0.1),
                        half_w, tbl_h,
                    )
                    # Reuse the same formatting loop via a local helper
                    _fill_table(tbl_shape.table, data, cfg, theme)
                    print(f"    Table (mixed): {n_rows} rows × {n_cols} cols")
                else:
                    add_table_slide(
                        slide, data, cfg,
                        slide_w, slide_h, content_top, theme,
                    )
            else:
                print(f"    ⚠  No data found in '{sheet_name}'")

        print(f"  ✓ Slide added: '{slide_title}'")

    prs.save(output_path)
    print(f"\n✓ Saved → {output_path}  ({len(prs.slides)} slides)")


# ── Internal helper used by mixed layout ────────────────────────────────────
def _fill_table(tbl, data, cfg, theme):
    """Apply text + formatting to an already-created pptx table object."""
    header = cfg.get("header_row", True)
    n_rows, n_cols = len(data), len(data[0])

    if n_rows > 25 or n_cols > 10:
        body_fs, hdr_fs = Pt(7), Pt(8)
    elif n_rows > 15 or n_cols > 7:
        body_fs, hdr_fs = Pt(9), Pt(10)
    else:
        body_fs, hdr_fs = Pt(11), Pt(12)

    accent  = RGBColor(*theme["accent"])
    hdr_txt = RGBColor(*theme["header_text"])
    even_bg = RGBColor(*theme["row_even"])
    odd_bg  = RGBColor(*theme["row_odd"])
    body_c  = RGBColor(*theme["cell_text"])

    for r, row_data in enumerate(data):
        is_header = header and r == 0
        bg = accent if is_header else (even_bg if r % 2 == 0 else odd_bg)
        fs = hdr_fs if is_header else body_fs

        for c, value in enumerate(row_data):
            if c >= tbl.columns.__len__():
                break
            cell = tbl.cell(r, c)
            cell.text = format_cell_value(value)
            cell.fill.solid()
            cell.fill.fore_color.rgb = bg
            tf = cell.text_frame
            tf.word_wrap = False
            for para in tf.paragraphs:
                para.alignment = PP_ALIGN.LEFT
                for run in para.runs:
                    run.font.size  = fs
                    run.font.bold  = is_header
                    run.font.name  = theme["body_font"]
                    run.font.color.rgb = hdr_txt if is_header else body_c

## ▶️ Run

In [ ]:
build_presentation(EXCEL_PATH, OUTPUT_PPTX, SHEET_CONFIG, THEME)

---
### Notes

| `content_type` | What happens |
|---|---|
| `"table"` | Reads cells → formatted PPTX table |
| `"chart"` | Extracts chart data via openpyxl → re-renders with matplotlib → embeds PNG |
| `"mixed"` | Table on left half, chart on right half |
| `"auto"` | Detects: chart if the sheet contains a chart object, otherwise table |

**Tips:**
- Set `data_range` to `"A1:F20"` to limit which cells are copied for table sheets.
- If a sheet has multiple charts, set `chart_index` to select which one.
- Chart colours, fonts, and the slide colour palette are all controlled through the `THEME` dict in the config cell.
- The workbook is opened with `data_only=True`, so formula cells show their **last-calculated value** (requires the file was saved with Excel after last edit).